## Мультиагентная система по образу общества Лумана для моделирования динамики реакции функциональной коммуникативной системы на шок

### Нет гейммастера. Результат д.б. в виде набора триплетов: (кто, что сделал, с чем)
смотрим набор мер от агентов разворачивающийся во времени

In [3]:
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from typing import List, Dict, Any, TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END
import json
import time

# Настройки API (заполните своими данными)
API_KEY = os.getenv("OPENAI_API_KEY")
URL = "https://foundation-models.api.cloud.ru/v1"
BASE_MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507"
RES_DIR = "c:/Users/PC1/Documents/PythonScripts/LLM CSS/Luhmann/Covid_experiments/РФ2020_5"
TEMPERATURE = 0.5
TARGET_YEAR = 5
INPUT_MSG = """Минздрав сообщил о первых случаях распространения нового штамма вируса внутри страны.
Заболевание может протекать как в форме острой респираторной вирусной инфекции лёгкого течения, так и в тяжёлой форме. 
К наиболее распространённым симптомам заболевания относятся повышенная температура тела, утомляемость и сухой кашель.
Распространяется воздушно-капельным путём через вдыхание распылённых в воздухе при кашле, чихании или разговоре капель с вирусом, 
а также через попадание вируса на поверхности с последующим занесением в глаза, нос или рот."""


#os.makedirs(os.path.dirname(RES_DIR), exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)


def create_agent_executor(system_prompt: str, tools: list):
    llm = ChatOpenAI(
        model=BASE_MODEL,
        api_key=API_KEY,
        base_url=URL,
        temperature=TEMPERATURE
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}")
    ])
    agent = create_openai_tools_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=True)
    return executor

# Промпты (замените пути на реальные)
with open('Промпты/Системный промпт политика_json_3.txt', encoding='utf-8') as f:
    POLITICS_PROMPT = f.read()
with open('Промпты/Системный промпт право_json_3.txt', encoding='utf-8') as f:
    LAW_PROMPT = f.read()
with open('Промпты/Системный промпт наука_json_3.txt', encoding='utf-8') as f:
    SCIENCE_PROMPT = f.read()
with open('Промпты/Системный промпт экономика_json_3.txt', encoding='utf-8') as f:
    ECONOMICS_PROMPT = f.read()
with open('Промпты/Системный промпт медиа_json_3.txt', encoding='utf-8') as f:
    MEDIA_PROMPT = f.read()


# Инструменты для отправки сообщений
@tool
def send_to_politics(message: str):
    """Отправить структурированное сообщение в подсистему politics.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: politics|{message}"

@tool
def send_to_law(message: str):
    """Отправить структурированное сообщение в подсистему law.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: law|{message}"

@tool
def send_to_science(message: str):
    """Отправить структурированное сообщение в подсистему science.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: science|{message}"

@tool
def send_to_media(message: str):
    """Отправить структурированное сообщение в подсистему media.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: media|{message}"

@tool
def send_to_economics(message: str):
    """Отправить структурированное сообщение в подсистему economics.
    Сообщение должно быть в формате JSON с полями:
    sender_subsystem, sender_org, recipient_subsystem, recipient_org, action."""
    return f"TOOL_MESSAGE: economics|{message}"


# Наборы инструментов для каждого агента
law_tools = [send_to_politics, send_to_economics, send_to_science, send_to_media]
politics_tools = [send_to_law, send_to_economics, send_to_science, send_to_media]
science_tools = [send_to_law, send_to_economics, send_to_media, send_to_politics]
economics_tools = [send_to_law, send_to_science, send_to_media, send_to_politics]
media_tools = [send_to_law, send_to_economics, send_to_science, send_to_politics]

# Создание агентов
law_agent = create_agent_executor(LAW_PROMPT, law_tools)
politics_agent = create_agent_executor(POLITICS_PROMPT, politics_tools)
science_agent = create_agent_executor(SCIENCE_PROMPT, science_tools)
economics_agent = create_agent_executor(ECONOMICS_PROMPT, economics_tools)
media_agent = create_agent_executor(MEDIA_PROMPT, media_tools)


def replace_year(current: int, new: int) -> int:
    return new

class AgentState(TypedDict):
    year: Annotated[int, replace_year]
    target_year: int
    history: Annotated[List[Dict[str, Any]], operator.add]
    agents_state: Dict[str, Any]
    # Два пула сообщений: текущий год и следующий год
    current_year_messages: List[Dict[str, str]]
    next_year_messages: List[Dict[str, str]]
    user_input: str
    output: str
    step_count: int


# Узел инициализации года
def distribute_cycle(state: AgentState) -> dict:
    updates = {
        "agents_state": {},
        "step_count": 0,
        "output": None
    }
    if state["year"] == 1:
        # В первый год добавляем стартовые сообщения в пул текущего года
        start_messages = [
            {
                "sender_subsystem": "system",
                "sender_org": "system",
                "recipient_subsystem": agent,
                "recipient_org": "system",
                "action": "START_YEAR"
            }
            for agent in ["law", "politics", "science", "economics", "media"]
        ]
        updates["current_year_messages"] = start_messages
    # Для последующих лет сообщения уже находятся в current_year_messages после finalize
    return updates


def extract_messages_from_output(output: str) -> List[Dict[str, str]]:
    import re
    import json
    pattern = r"TOOL_MESSAGE:\s*(\w+)\|(.*)"
    matches = re.findall(pattern, output.strip())
    messages = []
    for recipient, json_str in matches:
        # Исправление ошибки: отсутствие закрывающей скобки
        if json_str.count('{') > json_str.count('}'):
            json_str += '}'
        try:
            data = json.loads(json_str)
            required = ["sender_subsystem", "sender_org", "recipient_subsystem", "recipient_org", "action"]
            if all(k in data for k in required):
                messages.append(data)
            else:
                print(f"Пропущено сообщение: отсутствуют поля {required} в {data}")
        except json.JSONDecodeError:
            print(f"Не удалось распарсить JSON: {json_str}")
    return messages

def format_history(history: List[Dict]) -> List[Dict[str, str]]:
    formatted = []
    for entry in history[-5:]:
        formatted.append({"role": "user", "content": entry["input"]}) # сомнительно. Но как иначе? system?
        formatted.append({"role": "assistant", "content": entry["output"]})
    return formatted


# Фабрика узлов агентов
def make_agent_node(agent_name: str, agent_executor: AgentExecutor):
    def agent_node(state: AgentState) -> dict:
        # Сообщения, адресованные этому агенту, из пула текущего года
        my_messages = [msg for msg in state["current_year_messages"] if msg["recipient_subsystem"].lower() == agent_name]
        # Остальные сообщения остаются в текущем году для других агентов
        other_messages = [msg for msg in state["current_year_messages"] if msg["recipient_subsystem"] != agent_name]
        try:
            input_text = f"Год {state['year']}. "
            if my_messages:
                input_text += "Поступили сообщения:\n"
                for msg in my_messages:
                    input_text += f"- От {msg['sender_subsystem']}/{msg['sender_org']}: {msg['action']}\n"
            else:
                input_text += "Новых сообщений не поступило. Проведи оценку текущего состояния."
            # Механизм запуска шока
            if state.get("user_input"): 
                input_text += f"\nДополнительный контекст: {state['user_input']}"

            # показываем агенту входящую информацию.
            chat_history = format_history(state.get("history", [])[-5:]) # у каждого своя история из входящих и ответов
            result = agent_executor.invoke({
                "input": input_text,
                "chat_history": chat_history
            })

            new_messages = []
            # вычленяем сообщения отправленные другим агентам
            for action, observation in result.get("intermediate_steps", []):
                if isinstance(observation, str) and observation.lstrip().startswith("TOOL_MESSAGE:"):
                    new_messages.extend(extract_messages_from_output(observation))
            new_messages.extend(extract_messages_from_output(result["output"]))

            # Отправленные сообщения добавляем в пул следующего года
            current_next = state.get("next_year_messages", [])
            updated_next = current_next + new_messages

            return {
                "current_year_messages": other_messages,   # оставшиеся сообщения текущего года
                "next_year_messages": updated_next,        # новые сообщения для следующего года
                "history": [{
                    "agent": agent_name,
                    "input": input_text,
                    "output": result["output"] # реакция агента на данном шаге
                }],
                "agents_state": {agent_name: result.get("intermediate_steps", [])}
            }
        except Exception as e:
            print(f"Ошибка в агенте {agent_name}: {e}")
            return {"current_year_messages": other_messages}
    return agent_node


# Узлы агентов
law_node = make_agent_node("law", law_agent)
politics_node = make_agent_node("politics", politics_agent)
science_node = make_agent_node("science", science_agent)
economics_node = make_agent_node("economics", economics_agent)
media_node = make_agent_node("media", media_agent)


# Узел маршрутизации (просто увеличивает счётчик шагов) - теперь возвращает только обновление
def route_messages(state: AgentState) -> dict:
    return {"step_count": state.get("step_count", 0) + 1}

def route_decision(state: AgentState) -> str:
    if state.get("step_count", 0) > 20:          # защита от зацикливания
        return "finalize"
    if state["current_year_messages"]:
        # Берём первого получателя в списке (можно улучшить логику)
        recipient = state["current_year_messages"][0]["recipient_subsystem"].lower()
        return f"agent_{recipient}"
    else:
        return "finalize"


# Узел завершения года
def finalize(state: AgentState) -> dict:
    summary = f"Год {state['year']} завершён. Всего сообщений в истории: {len(state['history'])}"
    return {
        "year": state["year"] + 1,
        # Переносим сообщения следующего года в текущий год для нового цикла
        "current_year_messages": state["next_year_messages"],
        "next_year_messages": [],          # очищаем пул следующего года
        "output": summary,
        # history не возвращаем, чтобы не перезаписывать (используется add)
    }

def should_continue(state: AgentState) -> str:
    if state["year"] <= state["target_year"]:
        return "distribute"
    return END


# Построение графа
graph = StateGraph(AgentState)
graph.add_node("distribute", distribute_cycle)
graph.add_node("agent_law", law_node)
graph.add_node("agent_politics", politics_node)
graph.add_node("agent_science", science_node)
graph.add_node("agent_economics", economics_node)
graph.add_node("agent_media", media_node)
graph.add_node("route", route_messages)
graph.add_node("finalize", finalize)

graph.add_conditional_edges("finalize", should_continue)
graph.set_entry_point("distribute")
graph.add_edge("distribute", "route")
graph.add_conditional_edges("route", route_decision)

for agent in ["agent_law", "agent_politics", "agent_science", "agent_economics", "agent_media"]:
    graph.add_edge(agent, "route")

app = graph.compile()


# Запуск
initial_state = {
    "year": 1,
    "target_year": TARGET_YEAR,
    "user_input": INPUT_MSG,
    "agents_state": {},
    "current_year_messages": [],      # будет заполнено в distribute_cycle для года 1
    "next_year_messages": [],
    "history": [],
    "output": None,
    "step_count": 0
}

def save_full_history(history: List[Dict], filename="full_history.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"История сохранена в {filename} (записей: {len(history)})")

start = time.time()
for state in app.stream(initial_state, config={"recursion_limit": 100}, stream_mode="values"):
    #print(state)   # полное состояние после каждого узла

    # Если output установлен, значит только что отработал узел finalize – год завершён
    if state.get("output") is not None:
        year_completed = state["year"] - 1
        timestamp = time.strftime('%Y%m%d-%H%M%S')
        filename = f"history_year_{year_completed}_{timestamp}.json"
        filename = RES_DIR + "/" + filename
        save_full_history(state["history"], filename)

# Финальный вывод
if state.get("output") is not None:
    print("Финальный вывод:", state["output"])
else:
    print("Цикл завершился без финального вывода.")
ev_time = (time.time() - start) / 60
print(f"Оценка завершена за {ev_time:.2f} мин.")

C:\Users\PC1\anaconda3\envs\ml25_torch28_oai_lgrlch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: Function must have a docstring if description not provided.

In [9]:
RES_DIR = "c:/Users/PC1/Documents/PythonScripts/LLM CSS/Luhmann/Covid_experiments/РФ2000_4"
os.makedirs(os.path.dirname(RES_DIR), exist_ok=True)


In [1]:
def load_history(filename="full_history.json") -> List[Dict]:
    with open(filename, "r", encoding="utf-8") as f:
        return json.load(f)

def view_subsystem_history(history: List[Dict], subsystem: str):
    """
    Выводит все записи для заданной подсистемы (law, politics, science, economics, media).
    """
    entries = [e for e in history if e.get("agent") == subsystem]
    if not entries:
        print(f"Нет записей для подсистемы '{subsystem}'")
        return

    print(f"\n=== Подсистема: {subsystem.upper()} ===\n")
    for i, e in enumerate(entries, 1):
        # Извлекаем год из поля input (если оно начинается с "Год X.")
        year_info = e["input"].split(".")[0] if e["input"] else "?"
        print(f"[{i}] {year_info}")
        print(f"Входные данные:\n{e['input']}\n")
        print(f"Ответ агента:\n{e['output']}\n")
        print("-" * 80)

hist = load_history("history_year_1_20260308-175210.json")
view_subsystem_history(hist, "politics")
#view_subsystem_history(hist, "science")
#view_subsystem_history(hist, "media")

NameError: name 'List' is not defined

In [5]:
hist

[{'agent': 'law',
  'input': 'Год 3. Поступили сообщения:\n- От politics/Правительство: Направляет законопроект об экологических надбавках к налогу на прибыль для компаний, нарушающих международные обязательства по сохранению тропических лесов, с просьбой провести правовую экспертизу\n\nДополнительный контекст: Наука сообщает об ускорении вырубки лесов Амазонии.',
  'output': 'Год 3 завершён.\n\nПодсистема **Право** действовала в рамках своей операциональной замкнутости, реагируя на раздражение от политической подсистемы (запрос на экспертизу законопроекта) исключительно через призму кода **законно/незаконно**. \n\n### Ключевые итоги года:\n- Право **не оценивало** экологическую целесообразность или экономическую эффективность экологических надбавок — это вне его компетенции.\n- Право **не поддерживало и не отвергало** политическую цель законопроекта (сдерживание вырубки Амазонии), но **проверило его соответствие правовым стандартам**: определённость нормы, соблюдение процедур, констит

In [3]:
import json
import glob
import os
from typing import List, Dict

def merge_history_files(pattern: str = "history_year_*.json", output_file: str = "merged_history.json") -> List[Dict]:
    """
    Собирает все файлы, соответствующие pattern, загружает их содержимое (списки записей)
    и объединяет в один список. Сохраняет результат в output_file.
    Возвращает объединённый список.
    """
    all_entries = []
    files = sorted(glob.glob(pattern))  # сортировка по имени файла (предполагаем, что имена содержат дату)
    for fname in files:
        try:
            with open(fname, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    all_entries.extend(data)
                else:
                    print(f"Файл {fname} содержит не список, пропущен.")
        except Exception as e:
            print(f"Ошибка при чтении {fname}: {e}")

    # Дополнительно можно отсортировать записи по времени, если в них есть поле с timestamp
    # Но в текущей структуре такого поля нет, поэтому оставляем как есть.

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_entries, f, ensure_ascii=False, indent=2)

    print(f"Объединённая история сохранена в {output_file}. Всего записей: {len(all_entries)}")
    return all_entries

In [5]:
RES_DIR = "Covid_experiments/2"
merged = merge_history_files(RES_DIR+"/"+"history_year_*.json", RES_DIR+"/"+"covid_full_history.json")

Объединённая история сохранена в Covid_experiments/2/covid_full_history.json. Всего записей: 27


In [9]:
def view_subsystem_history(history: List[Dict], subsystem: str, year: int = None):
    """
    Выводит записи для заданной подсистемы. Если указан year, показывает только записи этого года.
    """
    entries = [e for e in history if e.get("agent") == subsystem]
    if year:
        # Извлекаем год из input (ожидается формат "Год X. ...")
        entries = [e for e in entries if e["input"].startswith(f"Год {year}.")]

    if not entries:
        print(f"Нет записей для подсистемы '{subsystem}'" + (f" за {year} год" if year else ""))
        return

    print(f"\n=== Подсистема: {subsystem.upper()} ===\n")
    for i, e in enumerate(entries, 1):
        year_info = e["input"].split(".")[0] if e["input"] else "?"
        print(f"[{i}] {year_info}")
        print(f"Входные данные:\n{e['input']}\n")
        print(f"Ответ агента:\n{e['output']}\n")
        print("-" * 80)

merged = merge_history_files()
view_subsystem_history(merged, "politics", year=3)

Объединённая история сохранена в Covid_experiments/2/merged_history.json. Всего записей: 27

=== Подсистема: POLITICS ===

[1] Год 3
Входные данные:
Год 3. Поступили сообщения:
- От science/Научный совет по инфекционным болезням: Предоставляет предварительную оценку рисков: высокий потенциал передачи, средняя тяжесть, рекомендует усиление эпиднадзора и подготовку к карантинным мерам
- От economics/Центральный банк: Предупреждение: введение карантина без компенсационных выплат приведёт к дефолтам в малом бизнесе и росту безработицы
- От media/Всероссийский телеканал "Новости": Запрос на участие представителя в прямом эфире для разъяснения мер мониторинга и готовности системы здравоохранения

Дополнительный контекст: Медики докладывают, о появлении нового опасного заболевания, 
которое может протекать как в форме острой респираторной вирусной инфекции лёгкого течения, так и в тяжёлой форме. 
К наиболее распространённым симптомам заболевания относятся повышенная температура тела, утомляем